# Survey Feedback Pipeline - Exploration

## Purpose
This notebook serves as an exploration and validation tool for the Survey Feedback Pipeline.
It demonstrates the data flow from raw CSVs through cleaning, transformation, and aggregation.

**What this notebook covers:**
1. Raw data exploration - understanding the source data
2. Data quality checks - identifying nulls, duplicates, and issues
3. Cleaning results - before/after comparison
4. Aggregation outputs - final metrics for analysis

**Note:** The production pipeline runs via Python scripts in the `src/` folder.
This notebook is for exploration, debugging, and presenting results.

## Setup

In [ ]:
import pandas as pd
import sys
import os

# add src to path (relative to notebook location)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
src_path = os.path.dirname(notebook_dir)  # go up one level from notebooks to src
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# run pipeline and get all data
from pipeline import run_pipeline
result = run_pipeline()

# unpack results
surveys_raw = result['surveys_raw']
users_raw = result['users_raw']
clean_surveys_df = result['surveys_clean']
clean_users_df = result['users_clean']
fct = result['fct']
aggs = result['aggs']

print('\nData loaded and ready!')

## 1. Raw Data Exploration

In [ ]:
# Raw data from pipeline
print('Raw Surveys:', len(surveys_raw), 'rows')
print('Raw Users:', len(users_raw), 'rows')

In [3]:
surveys_raw.head()

,submission_id,timestamp,user_email,rating,comment_text,region
0,a8d2b6ad-a8b9-4621-92cf-177a8b0d2c73,2024-01-09T02:22:24,liam.smith@examplecorp.com,4.0,Fantastic.,APAC
1,a2ca6df0-59c0-4311-a281-12694ddee7d7,2024-03-21T05:07:44,noah.lopez@examplecorp.com,4.0,Fantastic.,APAC
2,226a1d8b-342f-46df-aacb-bfc6f4142d84,2024-01-12T18:14:20,ava.garcia@examplecorp.com,5.0,Fantastic.,Americas
3,21504601-ce62-4c48-ae9b-42275732a10c,2024-01-15T08:47:31,sophia.johnson26@examplecorp.com,4.0,Great experience.,Americas
4,190c4dee-af10-4e70-a35b-fe38db104021,2024-01-05T03:25:06,sophia.nielsen@examplecorp.com,5.0,Smooth process.,EMEA


In [4]:
users_raw.head()

,user_email,full_name,department,country
0,olivia.nielsen@examplecorp.com,Olivia Nielsen,Marketing,Australia
1,liam.lopez@examplecorp.com,Liam Lopez,Customer Support,Germany
2,lucas.lopez@examplecorp.com,Lucas Lopez,Finance,Germany
3,sophia.garcia@examplecorp.com,Sophia Garcia,Product,United States
4,liam.garcia@examplecorp.com,Liam Garcia,Talent Acquisition,India


## 2. Data Quality Report

In [ ]:
print('DATA QUALITY REPORT')
print('=' * 40)

print('\nSURVEYS')
print(f'  Rows: {len(surveys_raw)}')
print(f'  Duplicates: {surveys_raw.duplicated(subset=["submission_id"]).sum()}')
print(f'  Nulls: {surveys_raw.isnull().sum().sum()} total')
for col in surveys_raw.columns:
    nulls = surveys_raw[col].isnull().sum()
    if nulls > 0:
        print(f'    - {col}: {nulls}')

print('\nUSERS')
print(f'  Rows: {len(users_raw)}')
print(f'  Duplicates: {users_raw.duplicated(subset=["user_email"]).sum()}')
print(f'  Nulls: {users_raw.isnull().sum().sum()} total')
for col in users_raw.columns:
    nulls = users_raw[col].isnull().sum()
    if nulls > 0:
        print(f'    - {col}: {nulls}')

## 3. Cleaned Data

In [ ]:
# Already loaded from pipeline
print('Before cleaning:', len(surveys_raw), 'surveys,', len(users_raw), 'users')
print('After cleaning:', len(clean_surveys_df), 'surveys,', len(clean_users_df), 'users')

## 3.1 Fact Table

The fact table joins surveys with user metadata and adds derived columns:

In [ ]:
print('Fact Table Columns:', list(fct.columns))
print('Rows:', len(fct))
print()
fct.head()

In [ ]:
# Rating category distribution (NPS-style)
print('Rating Categories:')
fct['rating_category'].value_counts()

In [ ]:
# Orphan surveys (no user match)
print('Orphan surveys:', fct['is_orphan'].sum())
fct[fct['is_orphan']][['submission_id', 'user_email', 'rating', 'department']]

## 4. Aggregation Results

In [ ]:
# Use aggregations from pipeline result
dept_avg = aggs['avg_rating_by_department']
region_avg = aggs['avg_rating_by_region']
dept_dist = aggs['rating_distribution_by_department']

print('Average Rating by Department:')
dept_avg

In [8]:
print('Average Rating by Region:')
region_avg

Average Rating by Region:


,region,avg_rating
0,APAC,3.815287
1,Americas,3.788618
2,EMEA,3.672131


In [9]:
print('Rating Distribution by Department:')
dept_dist

Rating Distribution by Department:


,department,rating,rating_count
0,Customer Support,1.0,4
1,Customer Support,2.0,9
2,Customer Support,3.0,16
3,Customer Support,4.0,32
4,Customer Support,5.0,24
5,Engineering,1.0,7
6,Engineering,2.0,10
7,Engineering,3.0,32
8,Engineering,4.0,57
9,Engineering,5.0,42


## 5. Visualizations

The charts below provide visual insights into the survey feedback data:

1. **Rating Distribution** - Shows how ratings are spread across 1-5 scale. Higher bars at 4-5 indicate overall positive feedback.

2. **Average Rating by Department** - Compares satisfaction across departments. Departments with lower averages may need attention.

3. **Average Rating by Region** - Compares regional performance. Useful for identifying geographic trends.

4. **Surveys by Region** - Shows survey volume distribution. Helps understand which regions have more respondents.

In [ ]:
import matplotlib.pyplot as plt

# Rating Distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Rating Distribution
ax1 = axes[0, 0]
surveys_raw['rating'].value_counts().sort_index().plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Rating Distribution')
ax1.set_xlabel('Rating')
ax1.set_ylabel('Count')

# 2. Average Rating by Department
ax2 = axes[0, 1]
dept_avg_sorted = dept_avg.sort_values('avg_rating')
ax2.barh(dept_avg_sorted['department'], dept_avg_sorted['avg_rating'], color='teal')
ax2.set_title('Average Rating by Department')
ax2.set_xlabel('Average Rating')
ax2.set_xlim(1, 5)

# 3. Average Rating by Region
ax3 = axes[1, 0]
ax3.bar(region_avg['region'], region_avg['avg_rating'], color='coral')
ax3.set_title('Average Rating by Region')
ax3.set_ylabel('Average Rating')
ax3.set_ylim(1, 5)

# 4. Surveys by Region
ax4 = axes[1, 1]
surveys_raw['region'].value_counts().plot(kind='pie', ax=ax4, autopct='%1.1f%%', colors=['steelblue', 'teal', 'coral'])
ax4.set_title('Surveys by Region')
ax4.set_ylabel('')

plt.tight_layout()
plt.show()